# 03 – Análise das Perguntas de Negócio e Geração de Dashboard

In [1]:
import sys
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))

## Objetivo

- Responder as perguntas do negocio com KPIs mensais e inadimplencia por maturidade (foco no recorrente).
- Gerar um dashboard HTML interativo em reports/dashboard.html.

## Saidas

- data/processed/kpis_monthly.csv
- data/processed/expected_vs_received.csv
- data/processed/default_rates.csv
- reports/dashboard.html

## Calculo de KPIs e persistencia

In [2]:
from pathlib import Path
import pandas as pd
import plotly.io as pio
from src.config import PROC_DIR, REPORTS_DIR, get_events
from src.metrics import monthly_sales_kpis, expected_vs_received, matured_default_rate_over_time, cohort_curves, segment_compare, interest_implied, interest_coverage
from src.viz import ts_line, line_two_axes, expected_received_area, cohort_area, interest_coverage_indicators, add_event_markers, default_rate_plot
events = get_events()
sales = pd.read_csv(PROC_DIR / 'sales_enriched.csv', parse_dates=['purchase_date'])
allocated = pd.read_csv(PROC_DIR / 'allocated.csv', parse_dates=['due_date','payment_date_eff'])
kpis = monthly_sales_kpis(sales)
evr = expected_vs_received(allocated)
defaults_t = matured_default_rate_over_time(allocated)
si = interest_implied(sales)
cov = interest_coverage(allocated, si, defaults_t['month'].max())
kpis.to_csv(PROC_DIR / 'kpis_monthly.csv', index=False)
evr.to_csv(PROC_DIR / 'expected_vs_received.csv', index=False)
defaults_t.to_csv(PROC_DIR / 'default_rates.csv', index=False)


## Dashboard (interativo)

In [3]:
fig_ticket = add_event_markers(line_two_axes(kpis, 'month', 'avg_ticket', 'contracts', 'Ticket médio', 'Contratos', 'Ticket médio e volume'), events)
fig_evr = add_event_markers(expected_received_area(evr, 'Receita esperada x recebida (área)'), events)
fig_def = add_event_markers(default_rate_plot(defaults_t, 'Inadimplência por maturidade'), events)
kpis2 = kpis.copy()
kpis2['recurring_share_pct'] = kpis2['recurring_share'] * 100
fig_rec_share = add_event_markers(ts_line(kpis2, 'month', 'recurring_share_pct', 'Participação de Vendas Recorrentes (%)', 'Recorrente (%)'), events)
evr2_plot = evr.copy()
evr2_plot['coverage_pct'] = (evr2_plot['received'] / evr2_plot['expected'].replace(0, pd.NA)) * 100
fig_cov = add_event_markers(ts_line(evr2_plot, 'month', 'coverage_pct', 'Cobertura de Recebimento (Recebido/Esperado %)', 'Cobertura (%)'), events)
fig_interest = interest_coverage_indicators(cov, 'Cobertura de Juros vs Perdas (contratos maduros)')
cohort_df = cohort_curves(allocated, sales)
cohort_df.to_csv(PROC_DIR / 'cohort_curves.csv', index=False)
fig_cohort = cohort_area(cohort_df, 'Cohort — Inadimplência vs Recorrente')

overview_parts = []
overview_parts.append(pio.to_html(fig_ticket, include_plotlyjs='cdn', full_html=False))
overview_parts.append(pio.to_html(fig_evr, include_plotlyjs=False, full_html=False))
overview_parts.append(pio.to_html(fig_def, include_plotlyjs=False, full_html=False))
overview_parts.append(pio.to_html(fig_rec_share, include_plotlyjs=False, full_html=False))
overview_parts.append(pio.to_html(fig_cov, include_plotlyjs=False, full_html=False))
overview_parts.append(pio.to_html(fig_interest, include_plotlyjs=False, full_html=False))
overview_html = ''.join(f"<div class='card'>{p}</div>" for p in overview_parts)

cohort_parts = [pio.to_html(fig_cohort, include_plotlyjs=False, full_html=False)]
cohort_html = ''.join(f"<div class='card'>{p}</div>" for p in cohort_parts)

dashboard = """<!doctype html><html lang='pt-br'><head><meta charset='utf-8'/><meta name='viewport' content='width=device-width, initial-scale=1'/>
<title>Dashboard – Tourism Company Delinquency</title>
<link rel='icon' href='favicon.ico'/>
<style>
body{background:#111;color:#eee;font-family:Inter,Arial,sans-serif;margin:0}
header{padding:24px;border-bottom:1px solid #333}
main{padding:24px;max-width:1200px;margin:0 auto}
.card{background:#1a1a1a;border:1px solid #2a2a2a;border-radius:12px;padding:16px;margin:16px 0}
.grid{display:grid;grid-template-columns:1fr;gap:16px}
@media(min-width:1000px){.grid{grid-template-columns:1fr 1fr}}
.tabs{display:flex;gap:8px;margin:16px 0}
.tab-btn{background:#1a1a1a;border:1px solid #2a2a2a;color:#eee;border-radius:10px;padding:10px 12px;cursor:pointer}
.tab-btn.active{border-color:#63b3ed}
.tab-panel{display:none}
.tab-panel.active{display:block}
</style>
</head><body>
<header><h1>Dashboard – Tourism Company Delinquency</h1></header>
<main>
<div class='tabs'>
  <button class='tab-btn active' data-tab='tab-overview'>Visão Geral</button>
  <button class='tab-btn' data-tab='tab-cohort'>Cohort</button>
</div>
<section id='tab-overview' class='tab-panel active'><div class='grid'>{overview}</div></section>
<section id='tab-cohort' class='tab-panel'><div class='grid'>{cohort}</div></section>
</main>
<script>
const btns = Array.from(document.querySelectorAll('.tab-btn'));
btns.forEach(b => b.addEventListener('click', () => {
  btns.forEach(x => x.classList.remove('active'));
  b.classList.add('active');
  const tabId = b.getAttribute('data-tab');
  document.querySelectorAll('.tab-panel').forEach(p => p.classList.remove('active'));
  document.getElementById(tabId).classList.add('active');
}));
</script>
</body></html>"""
dashboard = dashboard.replace('{overview}', overview_html).replace('{cohort}', cohort_html)
(REPORTS_DIR / 'dashboard_notebook.html').write_text(dashboard, encoding='utf-8')
events


{'high_ticket_start': datetime.date(2021, 1, 1),
 'recurring_expansion': datetime.date(2021, 7, 1),
 'premium_offer': datetime.date(2024, 8, 1),
 'whatsapp_start': datetime.date(2020, 2, 1)}

In [4]:
import json
from src.config import PROC_DIR

evr2 = evr.copy()
evr2['coverage'] = evr2['received'] / evr2['expected'].replace(0, pd.NA)

high_ticket = {
    'avg_ticket': segment_compare(kpis, pd.Timestamp(events['high_ticket_start']), 'avg_ticket'),
    'default': segment_compare(defaults_t, pd.Timestamp(events['high_ticket_start']), 'default_rate'),
    'receipt': segment_compare(evr2, pd.Timestamp(events['high_ticket_start']), 'coverage'),
}
recurring = {
    'contracts': segment_compare(kpis, pd.Timestamp(events['recurring_expansion']), 'contracts'),
    'default_rec': segment_compare(defaults_t, pd.Timestamp(events['recurring_expansion']), 'recurring_default_rate'),
    'receipt': segment_compare(evr2, pd.Timestamp(events['recurring_expansion']), 'coverage'),
}

whatsapp_default = {'pre': None, 'post': None, 'delta_abs': None, 'delta_pct': None}
if events.get('whatsapp_start'):
    whatsapp_default = segment_compare(defaults_t, pd.Timestamp(events['whatsapp_start']), 'default_rate')

premium_threshold = None
try:
    premium_threshold = float(pd.read_json(PROC_DIR / 'premium_threshold.json', typ='series').get('premium_threshold'))
except Exception:
    premium_threshold = None

premium = {
    'default': segment_compare(defaults_t, pd.Timestamp(events['premium_offer']), 'default_rate'),
    'receipt': segment_compare(evr2, pd.Timestamp(events['premium_offer']), 'coverage'),
}

cov_ratio = cov.get('coverage_ratio') if isinstance(cov, dict) else None
split = cov.get('split') if isinstance(cov, dict) else None
def fmt_x(v):
    return f"{float(v):.2f}x" if v is not None else "N/D"
interest = {
    'coverage_note': f"Cobertura estimada: {fmt_x(cov_ratio)}" if cov_ratio is not None else 'Cobertura não aplicável/insuficiente',
    'split_note': f"Recorrente: {fmt_x(split.get('recurring'))}; Tradicional: {fmt_x(split.get('traditional'))}" if isinstance(split, dict) else 'Sem divisão disponível',
}

context = {
    'high_ticket': high_ticket,
    'recurring': recurring,
    'whatsapp_start': str(events.get('whatsapp_start')) if events.get('whatsapp_start') else None,
    'whatsapp': {'default': whatsapp_default},
    'premium_threshold': premium_threshold,
    'premium': premium,
    'interest': interest,
    'executive_summary': 'Evidências quantitativas foram geradas para cada pergunta e estão refletidas no dashboard.',
}

(PROC_DIR / 'analysis_context.json').write_text(
    json.dumps(context, ensure_ascii=False, indent=2, default=str),
    encoding='utf-8'
)
context


{'high_ticket': {'avg_ticket': {'pre': 3638.3332666856018,
   'post': 6046.164624159336,
   'delta_abs': 2407.8313574737344,
   'delta_pct': 0.6617951630547543},
  'default': {'pre': 0.14552474204880622,
   'post': 0.4121030376141064,
   'delta_abs': 0.26657829556530016,
   'delta_pct': 1.8318417322869736},
  'receipt': {'pre': 0.32013252612006765,
   'post': 0.13546715898694892,
   'delta_abs': -0.18466536713311874,
   'delta_pct': -0.5768403772375784}},
 'recurring': {'contracts': {'pre': 896.7777777777778,
   'post': 2841.6481481481483,
   'delta_abs': 1944.8703703703704,
   'delta_pct': 2.1687316730681863},
  'default_rec': {'pre': 0.18142585523414864,
   'post': 0.42494915253931376,
   'delta_abs': 0.24352329730516512,
   'delta_pct': 1.3422744899886148},
  'receipt': {'pre': 0.28277690097723834,
   'post': 0.12973569911553068,
   'delta_abs': -0.15304120186170767,
   'delta_pct': -0.541208285870657}},
 'whatsapp_start': '2020-02-01',
 'whatsapp': {'default': {'pre': None,
   'pos